In [1646]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import torch
import torch.optim as optim

In [ ]:
from torch.utils.data import Dataset, DataLoader

In [1186]:
import torchmetrics

In [1148]:
df=pd.read_csv('online_gaming_behavior_dataset.csv')

In [1156]:
df1=df.copy()
df1.drop('PlayerID',axis=1,inplace=True)

In [1270]:
df['EngagementLevel'].value_counts(normalize=True)

2    0.483939
0    0.258181
1    0.257881
Name: EngagementLevel, dtype: float64

In [1159]:
df1=df1.join(pd.get_dummies(df1[['Gender','Location','GameGenre']],drop_first=True))


df1.drop(['Gender','Location','GameGenre'],axis=1,inplace=True)

In [1167]:
label_encoder = LabelEncoder()

# Fit and transform the 'color' column
df1['EngagementLevel'] = label_encoder.fit_transform(df1['EngagementLevel'])

In [1170]:
df1.columns

Index(['Age', 'PlayTimeHours', 'InGamePurchases', 'GameDifficulty',
       'SessionsPerWeek', 'AvgSessionDurationMinutes', 'PlayerLevel',
       'AchievementsUnlocked', 'EngagementLevel', 'Gender_Male',
       'Location_Europe', 'Location_Other', 'Location_USA', 'GameGenre_RPG',
       'GameGenre_Simulation', 'GameGenre_Sports', 'GameGenre_Strategy'],
      dtype='object')

In [1172]:
df1.columns=['Age', 'PlayTimeHours', 'InGamePurchases', 'GameDifficulty',
       'SessionsPerWeek', 'AvgSessionDurationMinutes', 'PlayerLevel',
       'AchievementsUnlocked','Gender_Male',
       'Location_Europe', 'Location_Other', 'Location_USA', 'GameGenre_RPG',
       'GameGenre_Simulation', 'GameGenre_Sports', 'GameGenre_Strategy','EngagementLevel' ]

In [1591]:
X=np.array(df1.drop('EngagementLevel',axis=1))
y=np.array(df1['EngagementLevel'])

In [1592]:
X.shape, y.shape

((40034, 16), (40034,))

In [1178]:
from torch import nn

device="cuda" if torch.cuda.is_available() else "cpu"

device

'cpu'

In [1461]:
def accuracy_fn(y_true,y_pred):
    correct=torch.eq(y_true, y_pred).sum().item() # torch.eq() calculates where two tensors are equal
    acc=(correct/len(y_pred))*100
    return acc

In [1462]:
precision = torchmetrics.Precision(task="multiclass", average='macro', num_classes=3)
recall = torchmetrics.Recall(task="multiclass",average='macro',num_classes=3)
f1_score = torchmetrics.F1Score(task="multiclass",average='macro',num_classes=3)
con_matrix=torchmetrics.ConfusionMatrix(task="multiclass",num_classes=3)

In [1678]:
class NeuralNetwork(nn.Module):
    def __init__(self,hidden_size):
        super().__init__()
#         hsize = trial.suggest_int(name="hidden_size", low=128, high=512, step=128)
        self.linear1=nn.Linear(16,hidden_size)
        self.linear2=nn.Linear(hidden_size,hidden_size)
        self.linear3=nn.Linear(hidden_size,3)
        self.relu=nn.Sigmoid()
        
    def forward(self,x):
        x=self.linear1(x)
        x=self.relu(x)
        x=self.linear2(x)
        x=self.relu(x)
        logits=self.linear3(x)
        return logits
    

In [1680]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test=train_test_split(X,y,test_size=0.2, random_state=42,shuffle=True,)  ## Make the random split reproducible

len(X_train),len(X_test), len(y_train),len(y_test)

(32027, 8007, 32027, 8007)

In [1681]:
X_train=pd.DataFrame(X_train,columns=df1.columns[:-1])
y_train=pd.DataFrame(y_train,columns=[df1.columns[-1]])

X_test=pd.DataFrame(X_test,columns=df1.columns[:-1])
y_test=pd.DataFrame(y_test,columns=[df1.columns[-1]])

df_train=X_train.join(y_train)


df_test=X_test.join(y_test)

In [1682]:

class DataFrameDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe
#         self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        sample = self.dataframe.iloc[idx, :-1].values # Features
        label = self.dataframe.iloc[idx, -1] # Label
        sample = torch.tensor(sample, dtype=torch.float32)
        label = torch.tensor(label, dtype=torch.float32)
        return sample, label

dataset_train = DataFrameDataset(df_train)
dataset_test = DataFrameDataset(df_test)

# Create Dataset and DataLoader
train_dataloader = DataLoader(dataset_train, batch_size=20, shuffle=True)
test_dataloader = DataLoader(dataset_test, batch_size=20, shuffle=True)

# # Iterate through the DataLoader
# for batch_idx, (batch_samples, batch_labels) in enumerate(train_dataloader):
#     print(f"Batch {batch_idx}:")
#     print("Features:", batch_samples)
#     print("Labels:", batch_labels)

In [1689]:
EPOCHS=10

In [1690]:
def objective(trial):

    # Generate the model.
    hidden_size=trial.suggest_int('hidden_size',384,512,128)
#                                   128,512,128)
    model = NeuralNetwork(hidden_size).to(device)

    # Generate the optimizers.
    optimizer_name = trial.suggest_categorical("optimizer", [#"Adam", "RMSprop",
                                                             "SGD"]) #for hp tuning

    lr = trial.suggest_float("lr", 0.01,0.9,step=0.01) #for hp tuning
 
    optimizer = getattr(optim, optimizer_name)(model.parameters(), lr=lr)
    CEloss = nn.CrossEntropyLoss() 


    # Training of the model.
    for epoch in range(EPOCHS):
        model.train()
        for batch_idx, (data, target) in enumerate(train_dataloader):
            
            data, target = data.to(device), target.to(device) 

            optimizer.zero_grad()
            output = model(data)
            loss = CEloss(output, target.long())
            loss.backward()
            optimizer.step()

        # Validation of the model.
        model.eval()
        correct = 0
        with torch.no_grad():
            val_loss_batch = 0
            for batch_idx, (data, target) in enumerate(test_dataloader):
                
                data, target = data.to(device), target.to(device) 
                output = model(data)
                # Get the index of the max log-probability.
                pred = output.argmax(dim=1, keepdim=True)
                correct += pred.eq(target.view_as(pred)).sum().item()

                val_loss_batch += CEloss(output, target.long()).item()  
        accuracy = correct / len(test_dataloader.dataset)


        trial.report(accuracy, epoch)
 

        # Handle pruning based on the intermediate value.
#         if trial.should_prune():
#             raise optuna.exceptions.TrialPruned()

    return accuracy 

In [1691]:
if __name__ == "__main__":
    study = optuna.create_study(direction="maximize")  # 'maximize' because objective function is returning accuracy
 
    study.optimize(objective, n_trials=30)

#     pruned_trials = [t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED]
#     complete_trials = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]

    print("Study statistics: ")
    print("  Number of finished trials: ", len(study.trials))
    print("Best trial:")
    trial = study.best_trial

    print("  Value: ", trial.value)

    print("  Params: ")
    for key, value in trial.params.items():
        print("    {}: {}".format(key, value))

[I 2025-06-15 12:53:40,349] A new study created in memory with name: no-name-26462ec7-31bc-4b55-9b94-d980b74ccbcb
C:\Users\Amrita\AppData\Local\Temp\ipykernel_9256\2297070755.py:4: FutureWarning:

suggest_int() got {'step'} as positional arguments but they were expected to be given as keyword arguments.

[I 2025-06-15 12:57:32,042] Trial 0 finished with value: 0.797677032596478 and parameters: {'hidden_size': 512, 'optimizer': 'SGD', 'lr': 0.8300000000000001}. Best is trial 0 with value: 0.797677032596478.
[I 2025-06-15 13:01:26,544] Trial 1 finished with value: 0.797677032596478 and parameters: {'hidden_size': 512, 'optimizer': 'SGD', 'lr': 0.8300000000000001}. Best is trial 0 with value: 0.797677032596478.
[I 2025-06-15 13:05:09,173] Trial 2 finished with value: 0.797677032596478 and parameters: {'hidden_size': 384, 'optimizer': 'SGD', 'lr': 0.7100000000000001}. Best is trial 0 with value: 0.797677032596478.
[I 2025-06-15 13:09:02,920] Trial 3 finished with value: 0.797677032596478 a

KeyboardInterrupt: 

In [1692]:
optuna.visualization.plot_optimization_history(study)

In [1693]:
optuna.visualization.plot_param_importances(study)

In [1694]:
optuna.visualization.plot_slice(study) 

In [1695]:
optuna.visualization.plot_parallel_coordinate(study)